In [ ]:
# =====================================================================
# BALANCED DATASET SOLUTION FOR BREAKHIS (IMBALANCED DATA FIX)
# Strategy: Augmentation-based Oversampling → Both classes same count
# Replace CELL 4 (Dataset) with this code
# =====================================================================

import os
import copy
import random
import numpy as np
from PIL import Image
from collections import Counter

import torch
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ── Config (same as original notebook) ────────────────────────────────
IMG_SIZE    = 224
SEED        = 42
BATCH_SIZE  = 16
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)


# =====================================================================
# STEP 1: Class Distribution Check  ──  কতটা imbalanced সেটা দেখো
# =====================================================================
def check_class_distribution(root_dir):
    """
    Dataset folder এ Benign vs Malignant কতটা imbalanced তা print করে।
    """
    class_counts = {}
    classes = sorted([
        d for d in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, d))
    ])
    for cls in classes:
        cls_dir = os.path.join(root_dir, cls)
        imgs = [
            f for f in os.listdir(cls_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff'))
        ]
        class_counts[cls] = len(imgs)

    print("=" * 50)
    print("  CLASS DISTRIBUTION (Before Balancing)")
    print("=" * 50)
    total = sum(class_counts.values())
    for cls, cnt in class_counts.items():
        pct = 100. * cnt / total
        bar = '█' * int(pct / 2)
        print(f"  {cls:<15}: {cnt:>5} images  ({pct:.1f}%)  {bar}")
    print(f"  {'TOTAL':<15}: {total:>5} images")
    majority   = max(class_counts, key=class_counts.get)
    minority   = min(class_counts, key=class_counts.get)
    ratio      = class_counts[majority] / class_counts[minority]
    print(f"\n  Imbalance Ratio: {ratio:.2f}x  "
          f"({majority} >> {minority})")
    print("=" * 50)
    return class_counts


# =====================================================================
# STEP 2: Heavy Augmentation Transform  ──  minority class এর জন্য
# =====================================================================

# Standard train transform (original notebook থেকে নেওয়া)
train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussNoise(p=0.2),
    A.ElasticTransform(p=0.2),
    A.GridDistortion(p=0.2),
    A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# Minority class এর জন্য EXTRA heavy augmentation
minority_heavy_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),

    # Spatial Transforms
    A.RandomRotate90(p=0.7),
    A.HorizontalFlip(p=0.6),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15,
                       rotate_limit=45, p=0.6),
    A.Transpose(p=0.3),

    # Color/Intensity Transforms (histopathology এ color stain variation)
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.15, p=0.6),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30,
                         val_shift_limit=20, p=0.5),
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.4),
    A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=0.3),

    # Noise / Blur
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.MotionBlur(blur_limit=5, p=0.2),

    # Deformation (tissue morphology variation)
    A.ElasticTransform(alpha=1.0, sigma=50, alpha_affine=50, p=0.3),
    A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3),
    A.OpticalDistortion(distort_limit=0.2, shift_limit=0.05, p=0.2),

    # Occlusion
    A.CoarseDropout(max_holes=10, max_height=20, max_width=20,
                    min_holes=3, p=0.4),
    A.RandomShadow(p=0.2),

    # Stain Augmentation (histopathology specific)
    A.RandomGamma(gamma_limit=(80, 120), p=0.3),

    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])


# =====================================================================
# STEP 3: Balanced Dataset Class  ──  Augmentation দিয়ে balance করে
# =====================================================================

class BalancedBreakHisDataset(Dataset):
    """
    Augmentation-based Oversampling দিয়ে class balance করে।

    Strategy:
    - Majority class: normal train_transform
    - Minority class: heavy augmentation দিয়ে oversample
    - উভয় class এ same number of images থাকবে

    Result: দুটো class তে same count → perfect balance
    """

    def __init__(self, root_dir, transform=None,
                 minority_transform=None, balance=True):
        self.root_dir           = root_dir
        self.balance            = balance

        classes = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])
        self.class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
        self.idx_to_class = {idx: cls for cls, idx in self.class_to_idx.items()}

        # Original images per class
        class_images = {cls: [] for cls in classes}
        for cls in classes:
            cls_dir = os.path.join(root_dir, cls)
            for img_name in os.listdir(cls_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff')):
                    class_images[cls].append(
                        os.path.join(cls_dir, img_name)
                    )

        # Majority / Minority চিহ্নিত করো
        counts       = {cls: len(imgs) for cls, imgs in class_images.items()}
        max_count    = max(counts.values())   # majority count
        self.majority_cls = max(counts, key=counts.get)
        self.minority_cls = min(counts, key=counts.get)

        print("\n" + "=" * 60)
        print("  BalancedBreakHisDataset — Augmentation Oversampling")
        print("=" * 60)
        for cls, cnt in counts.items():
            print(f"  Original  {cls:<15}: {cnt} images")

        # ── Build final image list ─────────────────────────────────
        self.images      = []
        self.labels      = []
        self.transforms  = []   # per-sample transform

        default_tf  = transform          or train_transform
        minority_tf = minority_transform or minority_heavy_transform

        if not balance:
            # No balancing: সব original images
            for cls in classes:
                for img_path in class_images[cls]:
                    self.images.append(img_path)
                    self.labels.append(self.class_to_idx[cls])
                    self.transforms.append(default_tf)
        else:
            # ── Majority class: original images ──────────────────
            for img_path in class_images[self.majority_cls]:
                self.images.append(img_path)
                self.labels.append(self.class_to_idx[self.majority_cls])
                self.transforms.append(default_tf)

            # ── Minority class: original + augmented copies ──────
            minority_idx   = self.class_to_idx[self.minority_cls]
            original_minority = class_images[self.minority_cls]
            n_orig         = len(original_minority)
            n_needed       = max_count  # majority count পর্যন্ত বাড়াবো

            # First: add all original minority images
            for img_path in original_minority:
                self.images.append(img_path)
                self.labels.append(minority_idx)
                self.transforms.append(default_tf)

            # Then: augmented copies to reach max_count
            n_aug      = n_needed - n_orig
            aug_pool   = original_minority * (n_aug // n_orig + 2)
            random.seed(SEED)
            random.shuffle(aug_pool)
            for img_path in aug_pool[:n_aug]:
                self.images.append(img_path)
                self.labels.append(minority_idx)
                self.transforms.append(minority_tf)   # heavy aug

        # ── Print final distribution ──────────────────────────────
        final_counts = Counter(self.labels)
        print("\n  After Balancing:")
        for idx, cnt in sorted(final_counts.items()):
            cls_name = self.idx_to_class[idx]
            print(f"  Balanced  {cls_name:<15}: {cnt} images")
        print(f"\n  Total images in dataset : {len(self.images)}")
        imbalance = abs(final_counts[0] - final_counts[1])
        print(f"  Class difference        : {imbalance} "
              f"({'BALANCED ✅' if imbalance == 0 else 'minor diff'})")
        print("=" * 60 + "\n")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img   = np.array(Image.open(self.images[idx]).convert('RGB'))
        label = self.labels[idx]
        tf    = self.transforms[idx]
        img   = tf(image=img)['image']
        return img, label

    @property
    def class_names(self):
        return [self.idx_to_class[i]
                for i in sorted(self.idx_to_class.keys())]


# =====================================================================
# STEP 4: Usage — Original Notebook এর CELL 4 replace করো
# =====================================================================

def create_balanced_dataloaders(train_path, test_path=None,
                                batch_size=BATCH_SIZE):
    """
    Balanced train + val dataloader তৈরি করে।
    Original notebook এর CELL 4 এর পরিবর্তে এটা ব্যবহার করো।
    """
    # Check distribution first
    print("📊 Checking original class distribution...")
    class_counts = check_class_distribution(train_path)

    # ── Full balanced dataset ──────────────────────────────────────
    full_dataset = BalancedBreakHisDataset(
        root_dir            = train_path,
        transform           = train_transform,
        minority_transform  = minority_heavy_transform,
        balance             = True,
    )

    all_labels = np.array(full_dataset.labels)
    all_images = full_dataset.images

    # ── Test set ───────────────────────────────────────────────────
    if test_path and os.path.exists(test_path):
        # Test set কে balance করার দরকার নেই — real distribution রাখো
        test_dataset = BalancedBreakHisDataset(
            root_dir  = test_path,
            transform = val_transform,
            balance   = False,   # test এ original distribution রাখো
        )
    else:
        # 10% holdout
        from torch.utils.data import random_split
        n_test    = max(1, int(0.1 * len(full_dataset)))
        n_train   = len(full_dataset) - n_test
        full_val  = BalancedBreakHisDataset(
            root_dir  = train_path,
            transform = val_transform,
            balance   = False,
        )
        _, test_dataset = random_split(
            full_val, [len(full_val) - n_test, n_test],
            generator=torch.Generator().manual_seed(SEED)
        )
        print("⚠️  Test folder not found → using 10% holdout from train.")

    test_loader = DataLoader(
        test_dataset, batch_size=batch_size,
        shuffle=False, num_workers=2, pin_memory=True
    )

    print(f"✅ Test samples   : {len(test_dataset)}")
    print(f"✅ Train samples  : {len(full_dataset)}")
    print(f"✅ DataLoaders ready!\n")

    return full_dataset, all_labels, all_images, test_loader


# =====================================================================
# STEP 5: KFold এর ভেতরে WeightedRandomSampler  (অতিরিক্ত safety)
# =====================================================================

def make_weighted_sampler(labels):
    """
    KFold এর প্রতিটা fold এ WeightedRandomSampler ব্যবহার করো।
    এটা augmentation balancing এর উপরে extra layer of protection।
    """
    label_arr    = np.array(labels)
    class_counts = np.bincount(label_arr)
    class_weights= 1.0 / class_counts.astype(float)
    sample_weights = class_weights[label_arr]
    sampler = WeightedRandomSampler(
        weights     = torch.FloatTensor(sample_weights),
        num_samples = len(sample_weights),
        replacement = True,
    )
    return sampler


# =====================================================================
# STEP 6: CELL 7B replacement  ──  Balanced KFold Loop
# =====================================================================

def run_balanced_kfold(train_path, n_folds, num_epochs,
                       batch_size, build_model_fn,
                       get_optimizer_scheduler_fn,
                       train_one_epoch_fn,
                       evaluate_fn,
                       compute_metrics_fn,
                       device,
                       class_names):
    """
    Balanced dataset দিয়ে KFold cross-validation চালাও।
    Original CELL 7B এর পরিবর্তে এটা ব্যবহার করো।
    """
    from sklearn.model_selection import StratifiedKFold

    # Balanced dataset তৈরি
    full_dataset, all_labels, _, test_loader = create_balanced_dataloaders(
        train_path, batch_size=batch_size
    )

    skf       = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    indices   = np.arange(len(all_labels))

    fold_histories = []
    fold_metrics   = []
    oof_probs      = np.zeros(len(all_labels))
    oof_preds      = np.zeros(len(all_labels), dtype=int)

    best_global_acc  = 0.0
    best_model_state = None

    for fold, (train_idx, val_idx) in enumerate(skf.split(indices, all_labels)):

        print(f"\n{'='*60}")
        # Per-fold class distribution check
        fold_train_labels = all_labels[train_idx]
        fold_val_labels   = all_labels[val_idx]
        train_dist = Counter(fold_train_labels.tolist())
        val_dist   = Counter(fold_val_labels.tolist())
        print(f"  FOLD {fold+1}/{n_folds}")
        print(f"  Train: {dict(train_dist)}  (total={len(train_idx)})")
        print(f"  Val  : {dict(val_dist)}    (total={len(val_idx)})")
        print(f"{'='*60}")

        # Subset datasets
        train_ds_fold = BalancedBreakHisDataset(
            train_path, transform=train_transform,
            minority_transform=minority_heavy_transform, balance=True
        )
        val_ds_fold = BalancedBreakHisDataset(
            train_path, transform=val_transform, balance=False
        )

        train_subset = Subset(train_ds_fold, train_idx)
        val_subset   = Subset(val_ds_fold,   val_idx)

        # WeightedSampler for extra safety
        fold_train_labels_sub = [train_ds_fold.labels[i] for i in train_idx]
        sampler = make_weighted_sampler(fold_train_labels_sub)

        train_loader_fold = DataLoader(
            train_subset, batch_size=batch_size,
            sampler=sampler,            # WeightedRandomSampler
            num_workers=2, pin_memory=True, drop_last=True
        )
        val_loader_fold = DataLoader(
            val_subset, batch_size=batch_size,
            shuffle=False, num_workers=2, pin_memory=True
        )

        model = build_model_fn()
        criterion, optimizer, scheduler = get_optimizer_scheduler_fn(
            model, len(train_loader_fold)
        )

        fold_hist = {
            'train_loss': [], 'val_loss': [],
            'train_acc' : [], 'val_acc' : [],
            'lr'        : []
        }
        best_fold_acc   = 0.0
        best_fold_state = None

        for epoch in range(num_epochs):
            tr_loss, tr_acc = train_one_epoch_fn(
                model, train_loader_fold, criterion, optimizer, scheduler, epoch
            )
            val_loss, val_acc, vp, vl, vprob = evaluate_fn(
                model, val_loader_fold, criterion
            )

            fold_hist['train_loss'].append(tr_loss)
            fold_hist['val_loss'].append(val_loss)
            fold_hist['train_acc'].append(tr_acc)
            fold_hist['val_acc'].append(val_acc)
            fold_hist['lr'].append(optimizer.param_groups[0]['lr'])

            print(f"  Epoch {epoch+1:02d}/{num_epochs} "
                  f"| TrLoss={tr_loss:.4f} TrAcc={tr_acc:.2f}% "
                  f"| ValLoss={val_loss:.4f} ValAcc={val_acc:.2f}%")

            if val_acc > best_fold_acc:
                best_fold_acc   = val_acc
                best_fold_state = copy.deepcopy(model.state_dict())

        model.load_state_dict(best_fold_state)
        _, _, oof_p, oof_l, oof_pr = evaluate_fn(
            model, val_loader_fold, criterion
        )
        oof_preds[val_idx] = oof_p
        oof_probs[val_idx] = oof_pr

        fold_m = compute_metrics_fn(oof_l, oof_p, oof_pr)
        fold_m['fold'] = fold + 1
        fold_metrics.append(fold_m)
        fold_histories.append(fold_hist)

        print(f"  ✅ Fold {fold+1} Best Val Acc = {best_fold_acc:.2f}% "
              f"| AUC = {fold_m['auc_roc']:.2f}%")

        if best_fold_acc > best_global_acc:
            best_global_acc  = best_fold_acc
            best_model_state = copy.deepcopy(best_fold_state)

    oof_metrics = compute_metrics_fn(all_labels, oof_preds, oof_probs)

    print("\n" + "="*60)
    print("  OVERALL CROSS-VALIDATION RESULTS (BALANCED)")
    print("="*60)
    for k, v in oof_metrics.items():
        print(f"  {k:<14}: {v:.4f}")

    return fold_metrics, fold_histories, oof_metrics, best_model_state, \
           best_global_acc, test_loader


# =====================================================================
# STEP 7: Balanced Loss Function  ──  CrossEntropyLoss with class weights
# =====================================================================

def get_balanced_criterion(train_labels, device, label_smoothing=0.1):
    """
    Class frequency থেকে automatic weight calculate করে।
    Minority class কে বেশি penalty দেয় training এ।
    """
    label_arr    = np.array(train_labels)
    class_counts = np.bincount(label_arr)
    # Inverse frequency weighting
    class_weights = 1.0 / class_counts.astype(float)
    class_weights = class_weights / class_weights.sum()   # normalize
    weight_tensor = torch.FloatTensor(class_weights).to(device)

    criterion = torch.nn.CrossEntropyLoss(
        weight=weight_tensor,
        label_smoothing=label_smoothing
    )
    print(f"  Class weights: {dict(enumerate(class_weights.round(4)))}")
    return criterion


# =====================================================================
# QUICK INTEGRATION GUIDE
# =====================================================================
"""
আপনার original notebook এ এই changes করুন:

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CELL 4 (Dataset) → নিচের code দিয়ে replace করুন:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

    full_dataset, all_labels, all_images, test_loader = \\
        create_balanced_dataloaders(TRAIN_PATH, TEST_PATH, BATCH_SIZE)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CELL 6 (get_optimizer_scheduler) → criterion replace করুন:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

    # পুরনো:
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # নতুন (balanced):
    criterion = get_balanced_criterion(all_labels, DEVICE)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CELL 7B (KFold loop) → DataLoader এ sampler যোগ করুন:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

    # প্রতিটা fold এ WeightedRandomSampler add করুন:
    sampler = make_weighted_sampler([train_ds_fold.labels[i]
                                     for i in train_idx])
    train_loader_fold = DataLoader(
        train_subset, batch_size=BATCH_SIZE,
        sampler=sampler,   # shuffle=True এর পরিবর্তে
        num_workers=2, pin_memory=True, drop_last=True
    )
"""


if __name__ == "__main__":
    # Quick test (BreaKHis path দিয়ে)
    TRAIN_PATH = '/kaggle/input/datasets/forderation/breakhis-400x/BreaKHis 400X/train'
    check_class_distribution(TRAIN_PATH)